In [ ]:
#Data Preprocessing

import pandas as pd
import numpy as np
from collections import deque
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. LOAD DATA
# ==========================================
data_train = pd.read_csv('train.csv')
data_test  = pd.read_csv('test.csv')

print("Data loaded")
print(f"   Train: {data_train.shape} | Test: {data_test.shape}")

# ==========================================
# 2. BASIC CLEANING
# ==========================================
data_train['date'] = pd.to_datetime(data_train['date'])
data_test['date']  = pd.to_datetime(data_test['date'])

for df in [data_train, data_test]:
    df['altitude_venue'] = df['altitude_venue'].replace(-9999, np.nan)

data_train = data_train.sort_values(['date', 'match_id']).reset_index(drop=True)
data_test  = data_test.sort_values(['date', 'match_id']).reset_index(drop=True)

cols_to_drop = [
    'team_avg_goals_last5', 'team_avg_conceded_last5', 'team_points_last5',
    'team_gd_last5', 'team_points_last10', 'team_win_rate_last10',
    'days_since_last_match_team',
    'opp_avg_goals_last5', 'opp_avg_conceded_last5', 'opp_points_last5',
    'opp_gd_last5', 'opp_points_last10', 'opp_win_rate_last10',
    'days_since_last_match_opp',
    # H2H raw juga di-drop, akan dihitung ulang
    'h2h_gd_last5', 'h2h_points_last5', 'h2h_missing_flag',
    'points_last5_diff', 'gd_last5_diff',
]
data_train = data_train.drop(columns=cols_to_drop, errors='ignore')
data_test  = data_test.drop(columns=cols_to_drop, errors='ignore')

print("Basic cleaning done")

# ==========================================
# 3. ELO ENGINE — DINAMIS
# ==========================================
def rank_to_initial_elo(rank):
    if pd.isna(rank) or rank <= 0:
        return 1200
    return max(1000, int(1950 - rank * 3.5))

def get_k_factor(tournament: str) -> float:
    t = str(tournament).strip()

    if 'World Cup' in t and 'qualification' not in t.lower():
        return 60.0
    if any(x in t for x in ['Asian Cup', 'African Cup', 'UEFA Euro',
                              'Copa América', 'Copa America', 'Gold Cup',
                              'Nations Cup', 'OFC Nations']):
        if 'qualification' not in t.lower():
            return 50.0
    if 'qualification' in t.lower() or 'qualifier' in t.lower():
        return 40.0
    if any(x in t for x in ['Nations League', 'Challenge Cup', 'Pan American',
                              'COSAFA', 'CECAFA', 'EAFF', 'AFF', 'SAFF', 'UNCAF']):
        return 35.0
    if 'friendly' in t.lower() or t == 'Friendly':
        return 20.0
    return 30.0

def get_goal_multiplier(goals_a: int, goals_b: int) -> float:
    gd = abs(goals_a - goals_b)
    if gd == 0:
        return 1.0
    elif gd == 1:
        return 1.0
    elif gd == 2:
        return 1.5
    else:
        return (11 + gd) / 8
elo_rating = {}

def init_elo_from_rank(team: str, rank: float):
    if team not in elo_rating:
        elo_rating[team] = rank_to_initial_elo(rank)

def get_elo(team: str) -> float:
    return elo_rating.get(team, 1200)

def update_elo(team_a: str, team_b: str,
               goals_a: int, goals_b: int,
               tournament: str,
               is_home_a: bool = False):
    Ra = get_elo(team_a)
    Rb = get_elo(team_b)

    HOME_ADVANTAGE = 100
    if is_home_a:
        Ra_adj = Ra + HOME_ADVANTAGE
    else:
        Ra_adj = Ra

    Ea = 1 / (1 + 10 ** ((Rb - Ra_adj) / 400))
    Eb = 1 - Ea

    if goals_a > goals_b:
        Sa, Sb = 1.0, 0.0
    elif goals_a < goals_b:
        Sa, Sb = 0.0, 1.0
    else:
        Sa, Sb = 0.5, 0.5

    K   = get_k_factor(tournament)
    GOL = get_goal_multiplier(goals_a, goals_b)

    elo_rating[team_a] = Ra + K * GOL * (Sa - Ea)
    elo_rating[team_b] = Rb + K * GOL * (Sb - Eb)

# ==========================================
# 4. FORM & H2H — SEQUENTIAL ENGINE
# ==========================================
form_history = {}

PRIOR = {
    'goals':      1.20,
    'conceded':   1.35,
    'points':     1.20,
    'win_rate':   0.38,
    'days':       None,
}

def init_form(team: str):
    if team not in form_history:
        form_history[team] = {
            'goals':    deque(maxlen=10),
            'conceded': deque(maxlen=10),
            'points':   deque(maxlen=10),
            'results':  deque(maxlen=10),
            'last_date': None,
        }

def safe_mean_prior(dq: deque, n: int, prior_val: float) -> float:
    arr = list(dq)[-n:] if n else list(dq)
    if len(arr) == 0:
        return prior_val
    weight = min(1.0, len(arr) / 5.0)
    return float(np.mean(arr)) * weight + prior_val * (1 - weight)

def get_form_features(team: str, current_date) -> dict:
    init_form(team)
    h = form_history[team]

    days_since = np.nan
    if h['last_date'] is not None:
        delta = (current_date - h['last_date']).days
        days_since = float(delta) if delta >= 0 else np.nan

    return {
        'team_avg_goals_last5':    safe_mean_prior(h['goals'],   5, PRIOR['goals']),
        'team_avg_conceded_last5': safe_mean_prior(h['conceded'],5, PRIOR['conceded']),
        'team_points_last5':       safe_mean_prior(h['points'],  5, PRIOR['points']),
        'team_gd_last5':           safe_mean_prior(h['goals'],   5, PRIOR['goals']) -
                                   safe_mean_prior(h['conceded'],5, PRIOR['conceded']),
        'team_points_last10':      safe_mean_prior(h['points'],  10, PRIOR['points']),
        'team_win_rate_last10':    safe_mean_prior(h['results'], 10, PRIOR['win_rate']),
        'days_since_last_match_team': days_since,
    }

def update_form(team: str, goals: int, conceded: int, date):
    init_form(team)
    h = form_history[team]
    pts = 3 if goals > conceded else (1 if goals == conceded else 0)
    res = 1.0 if goals > conceded else (0.5 if goals == conceded else 0.0)
    h['goals'].append(goals)
    h['conceded'].append(conceded)
    h['points'].append(pts)
    h['results'].append(res)
    h['last_date'] = date

h2h_history = {}

def get_h2h_key(team_a: str, team_b: str) -> str:
    return '|'.join(sorted([team_a, team_b]))

def init_h2h(team_a: str, team_b: str):
    key = get_h2h_key(team_a, team_b)
    if key not in h2h_history:
        h2h_history[key] = deque(maxlen=5)

def get_h2h_features(team: str, opponent: str) -> dict:
    key = get_h2h_key(team, opponent)
    if key not in h2h_history or len(h2h_history[key]) == 0:
        return {
            'h2h_points_last5': 0.0,
            'h2h_gd_last5':     0.0,
            'h2h_missing_flag': 1,
        }

    records = list(h2h_history[key])
    team_pts, team_gd = [], []

    for rec in records:
        if rec['home'] == team:
            g, c = rec['home_goals'], rec['away_goals']
        else:
            g, c = rec['away_goals'], rec['home_goals']

        pts = 3 if g > c else (1 if g == c else 0)
        team_pts.append(pts)
        team_gd.append(g - c)

    return {
        'h2h_points_last5': float(np.mean(team_pts)),
        'h2h_gd_last5':     float(np.mean(team_gd)),
        'h2h_missing_flag': 0,
    }

def update_h2h(team_a: str, team_b: str, goals_a: int, goals_b: int):
    key = get_h2h_key(team_a, team_b)
    init_h2h(team_a, team_b)
    h2h_history[key].append({
        'home': team_a, 'away': team_b,
        'home_goals': goals_a, 'away_goals': goals_b,
    })

# ==========================================
# 5. RANK ENGINE (sequential)
# ==========================================
rank_map = {}

def get_rank(team: str) -> float:
    return rank_map.get(team, np.nan)

def update_rank(team: str, rank: float):
    if not pd.isna(rank):
        rank_map[team] = rank

# ==========================================
# 6. GENERATE FEATURES TRAIN
# ==========================================
print("\nGenerating sequential features for TRAIN...")

for _, row in data_train.iterrows():
    team = row['team']
    rank = row.get('rank_team', np.nan)
    if team not in elo_rating:
        init_elo_from_rank(team, rank)

    opp  = row['opponent']
    rank_opp = row.get('rank_opponent', np.nan)
    if opp not in elo_rating:
        init_elo_from_rank(opp, rank_opp)

all_features = []

for match_id, group in data_train.groupby('match_id'):
    group = group.sort_values('date')
    if len(group) < 2:
        continue

    row1 = group.iloc[0]
    row2 = group.iloc[1]

    date     = row1['date']
    team1    = row1['team']
    team2    = row2['team']
    tourn    = row1.get('tournament', '')
    is_home1 = bool(row1.get('is_home', 0))

    elo1 = get_elo(team1)
    elo2 = get_elo(team2)

    r1_raw = row1.get('rank_team', np.nan)
    r2_raw = row2.get('rank_team', np.nan)
    r1 = r1_raw if not pd.isna(r1_raw) else get_rank(team1)
    r2 = r2_raw if not pd.isna(r2_raw) else get_rank(team2)

    form1 = get_form_features(team1, date)
    form2 = get_form_features(team2, date)

    h2h1 = get_h2h_features(team1, team2)
    h2h2 = get_h2h_features(team2, team1)

    def build_row(row, my_elo, opp_elo, my_rank, opp_rank,
                  my_form, opp_form, my_h2h, my_idx):
        opp_form_renamed = {
            k.replace('team_', 'opp_').replace('opp_opp_', 'opp_')
            if k != 'days_since_last_match_team'
            else 'days_since_last_match_opp': v
            for k, v in opp_form.items()
        }
        return {
            'index':         my_idx,
            'elo_team':      my_elo,
            'elo_opponent':  opp_elo,
            'rank_team':     my_rank,
            'rank_opponent': opp_rank,
            'rank_missing_team': int(pd.isna(my_rank)),
            'rank_missing_opp':  int(pd.isna(opp_rank)),
            **my_form,
            **opp_form_renamed,
            **my_h2h,
        }

    feat1 = build_row(row1, elo1, elo2, r1, r2, form1, form2, h2h1, row1.name)
    feat2 = build_row(row2, elo2, elo1, r2, r1, form2, form1, h2h2, row2.name)

    all_features.append(feat1)
    all_features.append(feat2)

    g1, c1 = int(row1['team_goals']), int(row1['opp_goals'])
    update_elo(team1, team2, g1, c1, tourn, is_home_a=is_home1)
    update_form(team1, g1, c1, date)
    update_form(team2, c1, g1, date)
    update_h2h(team1, team2, g1, c1)
    update_rank(team1, r1_raw)
    update_rank(team2, r2_raw)

feats_df = pd.DataFrame(all_features).set_index('index')

n_before = len(data_train)

cols_from_feats = [c for c in feats_df.columns if c in data_train.columns]
data_train = data_train.drop(columns=cols_from_feats, errors='ignore')

data_train = data_train.join(feats_df)
assert len(data_train) == n_before, f"Merge created duplicates! {len(data_train)} != {n_before}"
print(f"Train features generated ({len(all_features)} rows)")

# ==========================================
# 7. GENERATE FEATURES TEST
# ==========================================
print("Generating sequential features for TEST...")

for _, row in data_test.iterrows():
    for t_col, r_col in [('team', 'rank_team'), ('opponent', 'rank_opponent')]:
        team = row[t_col]
        if team not in elo_rating:
            init_elo_from_rank(team, row.get(r_col, np.nan))

all_features_test = []

for match_id, group in data_test.groupby('match_id'):
    group = group.sort_values('date')
    if len(group) < 2:
        continue

    row1 = group.iloc[0]
    row2 = group.iloc[1]

    date  = row1['date']
    team1 = row1['team']
    team2 = row2['team']

    elo1 = get_elo(team1)
    elo2 = get_elo(team2)

    r1 = row1.get('rank_team', np.nan)
    r2 = row2.get('rank_team', np.nan)
    if pd.isna(r1): r1 = get_rank(team1)
    if pd.isna(r2): r2 = get_rank(team2)

    form1 = get_form_features(team1, date)
    form2 = get_form_features(team2, date)
    h2h1  = get_h2h_features(team1, team2)
    h2h2  = get_h2h_features(team2, team1)

    def build_row_test(my_elo, opp_elo, my_rank, opp_rank,
                       my_form, opp_form, my_h2h, my_idx):
        opp_form_renamed = {
            k.replace('team_', 'opp_').replace('opp_opp_', 'opp_')
            if k != 'days_since_last_match_team'
            else 'days_since_last_match_opp': v
            for k, v in opp_form.items()
        }
        return {
            'index':         my_idx,
            'elo_team':      my_elo,
            'elo_opponent':  opp_elo,
            'rank_team':     my_rank,
            'rank_opponent': opp_rank,
            'rank_missing_team': int(pd.isna(my_rank)),
            'rank_missing_opp':  int(pd.isna(opp_rank)),
            **my_form,
            **opp_form_renamed,
            **my_h2h,
        }

    feat1 = build_row_test(elo1, elo2, r1, r2, form1, form2, h2h1, row1.name)
    feat2 = build_row_test(elo2, elo1, r2, r1, form2, form1, h2h2, row2.name)

    all_features_test.append(feat1)
    all_features_test.append(feat2)

feats_test_df = pd.DataFrame(all_features_test).set_index('index')

n_before = len(data_test)

cols_from_feats_test = [c for c in feats_test_df.columns if c in data_test.columns]
data_test = data_test.drop(columns=cols_from_feats_test, errors='ignore')

data_test = data_test.join(feats_test_df)
assert len(data_test) == n_before, f"Merge created duplicates! {len(data_test)} != {n_before}"
print(f"Test features generated ({len(all_features_test)} rows)")

# ==========================================
# 8. BASE FEATURES (elo_diff, rank_diff)
# ==========================================
for df in [data_train, data_test]:
    df['elo_diff']  = df['elo_team'] - df['elo_opponent']
    df['rank_diff'] = df['rank_team'] - df['rank_opponent']

# ==========================================
# 9. IMPUTE MISSING
# ==========================================
print("\nImputing missing values...")

max_rank = data_train[['rank_team', 'rank_opponent']].max().max()
FILL_RANK = max_rank + 10

for df in [data_train, data_test]:
    df['rank_team']     = df['rank_team'].fillna(FILL_RANK)
    df['rank_opponent'] = df['rank_opponent'].fillna(FILL_RANK)
    df['rank_diff']     = df['rank_team'] - df['rank_opponent']

DAYS_FILL = 999
for df in [data_train, data_test]:
    df['days_since_last_match_team'] = df['days_since_last_match_team'].fillna(DAYS_FILL)
    df['days_since_last_match_opp']  = df['days_since_last_match_opp'].fillna(DAYS_FILL)

team_map_pop  = data_train.groupby('team')['population_team'].median()
team_map_gdp  = data_train.groupby('team')['gdp_per_capita_team'].median()
opp_map_pop   = data_train.groupby('opponent')['population_opp'].median()
opp_map_gdp   = data_train.groupby('opponent')['gdp_per_capita_opp'].median()

for df in [data_train, data_test]:
    df['population_team']     = df['population_team'].fillna(df['team'].map(team_map_pop))
    df['gdp_per_capita_team'] = df['gdp_per_capita_team'].fillna(df['team'].map(team_map_gdp))
    df['population_opp']      = df['population_opp'].fillna(df['opponent'].map(opp_map_pop))
    df['gdp_per_capita_opp']  = df['gdp_per_capita_opp'].fillna(df['opponent'].map(opp_map_gdp))

for col in ['population_team', 'gdp_per_capita_team',
            'population_opp', 'gdp_per_capita_opp']:
    global_med = data_train[col].median()
    data_train[col] = data_train[col].fillna(global_med)
    data_test[col]  = data_test[col].fillna(global_med)

venue_alt_map  = data_train.groupby('venue_country')['altitude_venue'].median()
venue_temp_map = data_train.groupby('venue_country')['temperature_venue'].median()

for df in [data_train, data_test]:
    df['altitude_venue']    = df['altitude_venue'].fillna(df['venue_country'].map(venue_alt_map))
    df['temperature_venue'] = df['temperature_venue'].fillna(df['venue_country'].map(venue_temp_map))

for col in ['altitude_venue', 'temperature_venue']:
    global_med = data_train[col].median()
    data_train[col] = data_train[col].fillna(global_med)
    data_test[col]  = data_test[col].fillna(global_med)

def impute_distance(df_target, df_source, col_dist, col_entity, col_venue):
    """Hierarchical distance imputation dengan safety check."""
    n_before = len(df_target)

    mapping = (
        df_source.groupby([col_entity, col_venue])[col_dist]
        .median()
        .reset_index()
        .rename(columns={col_dist: '_dist_fill'})
    )
    df_target = df_target.merge(mapping, on=[col_entity, col_venue], how='left')
    df_target[col_dist] = df_target[col_dist].fillna(df_target['_dist_fill'])
    df_target.drop(columns=['_dist_fill'], inplace=True)

    assert len(df_target) == n_before, f"Distance merge duplicated rows for {col_dist}!"

    entity_map = df_source.groupby(col_entity)[col_dist].median()
    df_target[col_dist] = df_target[col_dist].fillna(df_target[col_entity].map(entity_map))

    df_target[col_dist] = df_target[col_dist].fillna(df_source[col_dist].median())

    return df_target

data_train = impute_distance(data_train, data_train, 'distance_travel_team', 'team', 'venue_country')
data_train = impute_distance(data_train, data_train, 'distance_travel_opp', 'opponent', 'venue_country')
data_test  = impute_distance(data_test,  data_train, 'distance_travel_team', 'team', 'venue_country')
data_test  = impute_distance(data_test,  data_train, 'distance_travel_opp',  'opponent', 'venue_country')

print("Missing value imputation done")

# ==========================================
# 10. FINAL MISSING VALUE CHECK
# ==========================================
def cek_kebersihan(df, nama):
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"\n========== Cek Kebersihan: {nama} ==========")
    if len(missing) == 0:
        print("Tidak ada data kosong.")
    else:
        pct = (missing / len(df) * 100).round(2)
        print(pd.concat([missing, pct], axis=1, keys=['Total', '%']).sort_values('%', ascending=False))

    dupes = df.duplicated().sum()
    print(f"Duplikat: {dupes}")
    print("=" * 50)

cek_kebersihan(data_train, "Train")
cek_kebersihan(data_test,  "Test")

# ==========================================
# 11. ELO DISTRIBUTION CHECK
# ==========================================
print("\nELO Distribution Check:")
print(f"   Train elo_team  — mean: {data_train['elo_team'].mean():.1f} "
      f"std: {data_train['elo_team'].std():.1f} "
      f"min: {data_train['elo_team'].min():.1f} "
      f"max: {data_train['elo_team'].max():.1f}")
print(f"   Test  elo_team  — mean: {data_test['elo_team'].mean():.1f} "
      f"std: {data_test['elo_team'].std():.1f}")

elo_std = data_train['elo_team'].std()
if elo_std < 100:
    print("ELO std < 100 — mungkin perlu lebih banyak pertandingan untuk konvergen")
elif elo_std > 400:
    print("ELO std > 400 — mungkin ada outlier ekstrem")
else:
    print("ELO distribution terlihat sehat")

# ==========================================
# 12. SAVE
# ==========================================
data_train.to_csv('data_train_clean_fix.csv', index=False)
data_test.to_csv('data_test_clean_fix.csv', index=False)

print(f"\n'data_train_clean_fix.csv' saved — {data_train.shape}")
print(f"'data_test_clean_fix.csv'  saved — {data_test.shape}")

# ==========================================
# 13. FEATURE SUMMARY
# ==========================================
new_cols = [
    'elo_team', 'elo_opponent', 'elo_diff',
    'rank_team', 'rank_opponent', 'rank_diff',
    'rank_missing_team', 'rank_missing_opp',
    'team_avg_goals_last5', 'team_avg_conceded_last5',
    'team_points_last5', 'team_gd_last5',
    'team_points_last10', 'team_win_rate_last10',
    'days_since_last_match_team',
    'opp_avg_goals_last5', 'opp_avg_conceded_last5',
    'opp_points_last5', 'opp_gd_last5',
    'opp_points_last10', 'opp_win_rate_last10',
    'days_since_last_match_opp',
    'h2h_points_last5', 'h2h_gd_last5', 'h2h_missing_flag',
]

print("\nNew feature columns generated:")
for col in new_cols:
    avail_train = "✅" if col in data_train.columns else "❌"
    avail_test  = "✅" if col in data_test.columns  else "❌"
    print(f"   {avail_train} train | {avail_test} test | {col}")




Data loaded
   Train: (78772, 47) | Test: (42422, 20)
Basic cleaning done

Generating sequential features for TRAIN...
Train features generated (78772 rows)
Generating sequential features for TEST...
Test features generated (42422 rows)

Imputing missing values...
Missing value imputation done

========== Cek Kebersihan: Train ==========
Tidak ada data kosong.
Duplikat: 0

========== Cek Kebersihan: Test ==========
Tidak ada data kosong.
Duplikat: 0

ELO Distribution Check:
   Train elo_team  — mean: 1301.0 std: 218.3 min: 592.2 max: 2261.3
   Test  elo_team  — mean: 1324.8 std: 335.3
ELO distribution terlihat sehat

'data_train_clean_fix.csv' saved — (78772, 47)
'data_test_clean_fix.csv'  saved — (42422, 45)

New feature columns generated:
   ✅ train | ✅ test | elo_team
   ✅ train | ✅ test | elo_opponent
   ✅ train | ✅ test | elo_diff
   ✅ train | ✅ test | rank_team
   ✅ train | ✅ test | rank_opponent
   ✅ train | ✅ test | rank_diff
   ✅ train | ✅ test | rank_missing_team
   ✅ train |

In [ ]:
#Exploratory Data Analysis

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

cl_train = pd.read_csv('data_train_clean_fix.csv')
cl_test  = pd.read_csv('data_test_clean_fix.csv')
cl_train['date'] = pd.to_datetime(cl_train['date'])
tr93 = cl_train[cl_train['date'].dt.year >= 1993].copy()

fig = plt.figure(figsize=(20, 24))
fig.suptitle('Football Score Prediction — EDA & Distribution Analysis',
             fontsize=16, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
scores = {}
for t in range(6):
    for o in range(6):
        p = ((tr93['team_goals']==t)&(tr93['opp_goals']==o)).mean()
        if p > 0.01:
            scores[f"{t}-{o}"] = p
top_scores = sorted(scores.items(), key=lambda x: -x[1])[:12]
labels, vals = zip(*top_scores)
bars = ax1.barh(range(len(labels)), vals, color=plt.cm.RdYlGn(np.linspace(0.3,0.9,len(labels))))
ax1.set_yticks(range(len(labels))); ax1.set_yticklabels(labels, fontsize=9)
ax1.set_xlabel('Probability'); ax1.set_title('Top 12 Score Outcomes\n(Train post-1993)', fontweight='bold')
for i, v in enumerate(vals):
    ax1.text(v+0.001, i, f'{v:.1%}', va='center', fontsize=8)

ax2 = fig.add_subplot(gs[0, 1])
w = (tr93['team_goals'] > tr93['opp_goals']).mean()
d = (tr93['team_goals'] == tr93['opp_goals']).mean()
l = (tr93['team_goals'] < tr93['opp_goals']).mean()
wedges, texts, autotexts = ax2.pie([w,d,l], labels=['Win\n(39.3%)', 'Draw\n(21.5%)', 'Loss\n(39.3%)'],
    colors=['#2ecc71','#f39c12','#e74c3c'], autopct='%1.1f%%', startangle=90, pctdistance=0.7)
ax2.set_title('Outcome Distribution\n(Train post-1993)', fontweight='bold')

ax3 = fig.add_subplot(gs[0, 2])
goals_range = range(9)
team_dist = [(tr93['team_goals']==g).mean() for g in goals_range]
opp_dist  = [(tr93['opp_goals']==g).mean() for g in goals_range]
x = np.arange(len(goals_range))
ax3.bar(x-0.2, team_dist, 0.4, label='Team', color='#3498db', alpha=0.8)
ax3.bar(x+0.2, opp_dist,  0.4, label='Opp',  color='#e74c3c', alpha=0.8)
ax3.set_xticks(x); ax3.set_xticklabels(goals_range)
ax3.set_xlabel('Goals'); ax3.set_ylabel('Probability')
ax3.set_title('Goals Distribution\n(symmetric as expected)', fontweight='bold')
ax3.legend()

ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(tr93['elo_diff'], bins=60, density=True, alpha=0.6, color='#3498db', label='Train')
ax4.hist(cl_test['elo_diff'], bins=60, density=True, alpha=0.6, color='#e67e22', label='Test')
ax4.axvline(0, color='black', linestyle='--', linewidth=1)
ax4.set_xlabel('ELO Diff'); ax4.set_title('ELO Diff Distribution\nTrain vs Test', fontweight='bold')
ax4.legend()
std_tr = tr93['elo_diff'].std(); std_te = cl_test['elo_diff'].std()
ax4.text(0.05, 0.95, f"Train std={std_tr:.0f}\nTest  std={std_te:.0f}",
         transform=ax4.transAxes, va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax5 = fig.add_subplot(gs[1, 1])
form_cols = ['team_avg_goals_last5','team_avg_conceded_last5','opp_avg_goals_last5','opp_avg_conceded_last5']
tr_means = [tr93[c].mean() for c in form_cols]
te_means = [cl_test[c].mean() for c in form_cols]
short = ['tg5','tc5','og5','oc5']
x = np.arange(len(form_cols))
ax5.bar(x-0.2, tr_means, 0.4, label='Train', color='#3498db', alpha=0.8)
ax5.bar(x+0.2, te_means, 0.4, label='Test',  color='#e67e22', alpha=0.8)
ax5.set_xticks(x); ax5.set_xticklabels(short, fontsize=9)
ax5.set_title('Form Features: Train vs Test\n⚠️ tc5 shift +0.60!', fontweight='bold', color='red')
ax5.legend()
ax5.axhline(1.563, color='gray', linestyle=':', linewidth=1, label='True mean')

ax6 = fig.add_subplot(gs[1, 2])
ax6.hist(tr93['days_since_last_match_team'].clip(0,365), bins=50, density=True,
         alpha=0.7, color='#3498db', label='Train (clipped 365)')
ax6.hist(cl_test['days_since_last_match_team'].clip(0,5000), bins=50, density=True,
         alpha=0.7, color='#e74c3c', label='Test (mean=3132!)')
ax6.set_xlabel('Days'); ax6.set_title('Days Since Last Match\n⚠️ Test has INVALID values (mean=3132)',
                                        fontweight='bold', color='red')
ax6.legend(fontsize=8)

ax7 = fig.add_subplot(gs[2, 0])
tr93['elo_prob'] = 1 / (1 + 10**(-(tr93['elo_diff']/400)))
tr93['actual_win'] = (tr93['team_goals'] > tr93['opp_goals']).astype(float)
bins = np.linspace(0,1,11)
bin_centers = (bins[:-1]+bins[1:])/2
actual_wr = []
for i in range(len(bins)-1):
    mask = (tr93['elo_prob']>=bins[i]) & (tr93['elo_prob']<bins[i+1])
    actual_wr.append(tr93.loc[mask,'actual_win'].mean() if mask.sum()>10 else np.nan)
ax7.plot(bin_centers, bin_centers, 'k--', label='Perfect calibration', linewidth=1.5)
ax7.plot(bin_centers, actual_wr, 'ro-', label='Actual', linewidth=2, markersize=6)
ax7.fill_between(bin_centers, bin_centers, actual_wr, alpha=0.2, color='red')
ax7.set_xlabel('ELO Win Prob'); ax7.set_ylabel('Actual Win Rate')
ax7.set_title('ELO Calibration Curve\n(over-confident for strong teams)', fontweight='bold')
ax7.legend(); ax7.set_xlim(0,1); ax7.set_ylim(0,1)

ax8 = fig.add_subplot(gs[2, 1])
top_t = tr93['tournament'].value_counts(normalize=True).head(10)
ax8.barh(range(len(top_t)), top_t.values, color='#9b59b6', alpha=0.8)
ax8.set_yticks(range(len(top_t))); ax8.set_yticklabels([t[:30] for t in top_t.index], fontsize=8)
ax8.set_title('Top 10 Tournaments (Train)', fontweight='bold')
ax8.set_xlabel('Proportion')

ax9 = fig.add_subplot(gs[2, 2])
cl_train['date'] = pd.to_datetime(cl_train['date'])
cl_train['year'] = cl_train['date'].dt.year
elo_std_by_year = cl_train.groupby('year')['elo_diff'].std()
elo_std_by_year[elo_std_by_year.index >= 1990].plot(ax=ax9, color='#1abc9c', linewidth=2)
ax9.axvline(2011, color='red', linestyle='--', linewidth=2, label='Train/Test split')
ax9.set_title('ELO Diff Std by Year\n(Test era: higher variance)', fontweight='bold')
ax9.set_xlabel('Year'); ax9.set_ylabel('Std(ELO diff)')
ax9.legend()

ax10 = fig.add_subplot(gs[3, 0])
heatmap_data = np.zeros((6,6))
for t in range(6):
    for o in range(6):
        heatmap_data[t,o] = ((tr93['team_goals']==t)&(tr93['opp_goals']==o)).mean()*100
im = ax10.imshow(heatmap_data, cmap='YlOrRd')
ax10.set_xticks(range(6)); ax10.set_yticks(range(6))
ax10.set_xticklabels(range(6)); ax10.set_yticklabels(range(6))
ax10.set_xlabel('Opp Goals'); ax10.set_ylabel('Team Goals')
ax10.set_title('Score Heatmap (%)\n(Train post-1993)', fontweight='bold')
for t in range(6):
    for o in range(6):
        ax10.text(o, t, f'{heatmap_data[t,o]:.1f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax10)

ax11 = fig.add_subplot(gs[3, 1])
versions = ['v29','v30a','v30b','v31','v32','v28']
scores_kaggle = [3.288, 3.507, 3.471, 3.412, 3.330, 3.290]
colors = ['#e74c3c' if s > 3.35 else '#f39c12' if s > 3.3 else '#2ecc71' for s in scores_kaggle]
bars2 = ax11.bar(versions, scores_kaggle, color=colors, alpha=0.85)
ax11.axhline(3.0, color='green', linestyle='--', linewidth=2, label='Target: 3.0')
ax11.axhline(3.289, color='orange', linestyle=':', linewidth=1.5, label='Best so far')
ax11.set_ylim(3.1, 3.6); ax11.set_ylabel('AW-MAE')
ax11.set_title('Kaggle Score History\n(lower = better)', fontweight='bold')
ax11.legend(fontsize=8)
for bar, score in zip(bars2, scores_kaggle):
    ax11.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{score:.3f}',
              ha='center', va='bottom', fontsize=9, fontweight='bold')

ax12 = fig.add_subplot(gs[3, 2])
pred_types = ['Correct\nOutcome\nExact Score', 'Correct\nOutcome\nWrong Score',
              'Wrong\nOutcome\n1 goal off', 'Wrong\nOutcome\n2 goals off']
def loss_example(mae, exact, ok, gd_ok, mult=None):
    if mult is None: mult = 1.0 if ok else 1.5
    pen = 0.30*(1-exact) + 0.25*(1-ok) + 0.15*(1-gd_ok)
    return ((mae+pen)*mult)**1.5

examples = [
    loss_example(0, 1, 1, 1),
    loss_example(1, 0, 1, 0),
    loss_example(0.5, 0, 0, 0),
    loss_example(1.5, 0, 0, 0),
]
colors_ex = ['#2ecc71','#f1c40f','#e67e22','#e74c3c']
bars3 = ax12.bar(pred_types, examples, color=colors_ex, alpha=0.85)
ax12.set_ylabel('Loss per match'); ax12.set_title('AW-MAE Loss by Prediction Type\n(penalty structure)', fontweight='bold')
ax12.tick_params(axis='x', labelsize=8)
for bar, val in zip(bars3, examples):
    ax12.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.3f}',
              ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.savefig('EDA.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
print("EDA visualization saved!")


EDA visualization saved!


In [ ]:
#Modelling

import pandas as pd
import numpy as np
import xgboost as xgb
import math
import warnings
from collections import Counter
warnings.filterwarnings("ignore")

# ==========================================
# 1. LOAD & REFINED WEIGHTING
# ==========================================
train = pd.read_csv("data_train_clean_fix.csv")
test  = pd.read_csv("data_test_clean_fix.csv")

train['date'] = pd.to_datetime(train['date'])
train = train.sort_values('date').reset_index(drop=True)
train = train[train['date'].dt.year >= 1993].reset_index(drop=True)

def get_weight(t):
    t = str(t)
    if 'World Cup' in t and 'qualification' not in t: return 3.0
    elif 'World Cup' in t: return 2.2
    elif 'qualification' in t: return 1.8
    elif 'Friendly' in t: return 0.75
    return 1.2

train['w'] = train['tournament'].apply(get_weight)
yr = train['date'].dt.year
rec = np.exp(3.0 * (yr - yr.min()) / (yr.max() - yr.min()))
weights = (train['w'] * (rec / rec.mean())).values

train['outcome'] = np.where(train['team_goals'] > train['opp_goals'], 2,
                   np.where(train['team_goals'] == train['opp_goals'], 1, 0))

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================
def assign_tier(elo):
    if   elo >= 1950: return 5
    elif elo >= 1800: return 4
    elif elo >= 1650: return 3
    elif elo >= 1500: return 2
    else:             return 1

def fe_v29(df):
    df = df.copy()
    df['elo_diff'] = df['elo_team'] - df['elo_opponent']
    df['rank_diff'] = (df['rank_team'] - df['rank_opponent']).clip(-150, 150)
    df['tier_team'] = df['elo_team'].apply(assign_tier)
    df['tier_opp']  = df['elo_opponent'].apply(assign_tier)
    df['tier_diff'] = df['tier_team'] - df['tier_opp']

    df['team_momentum'] = df['team_avg_goals_last5'] - df['team_avg_conceded_last5']
    df['opp_momentum'] = df['opp_avg_goals_last5'] - df['opp_avg_conceded_last5']
    df['momentum_diff'] = df['team_momentum'] - df['opp_momentum']

    df['xg_team'] = np.sqrt(np.clip(df['team_avg_goals_last5'] * df['opp_avg_conceded_last5'], 0, None))
    df['xg_opp']  = np.sqrt(np.clip(df['opp_avg_goals_last5'] * df['team_avg_conceded_last5'], 0, None))
    df['xg_diff'] = df['xg_team'] - df['xg_opp']

    df['rest_diff'] = (df['days_since_last_match_team'] - df['days_since_last_match_opp']).clip(-14, 14)
    df['travel_diff'] = (df['distance_travel_team'] - df['distance_travel_opp']).clip(-10000, 10000)

    df['neutral_rank_int'] = df['neutral'] * df['rank_diff']
    df['home_elo_int'] = df['is_home'] * df['elo_diff']
    return df

train = fe_v29(train)
test  = fe_v29(test)

FEATS = ['elo_diff', 'rank_diff', 'tier_diff', 'momentum_diff', 'xg_team', 'xg_opp',
         'xg_diff', 'rest_diff', 'travel_diff', 'neutral_rank_int', 'home_elo_int', 'is_home', 'neutral']

X = train[FEATS].fillna(0).reset_index(drop=True)
Xt = test[FEATS].fillna(0).reset_index(drop=True)
y_t, y_o, y_out = train['team_goals'].clip(0,8), train['opp_goals'].clip(0,8), train['outcome']

# ==========================================
# 3. TRAINING
# ==========================================
print("Training Models v29.1...")
clf = xgb.XGBClassifier(max_depth=5, learning_rate=0.012, n_estimators=2800, subsample=0.8,
                        colsample_bytree=0.75, objective='multi:softprob', num_class=3,
                        tree_method='hist', random_state=42).fit(X, y_out, sample_weight=weights)

mt = xgb.XGBRegressor(max_depth=4, learning_rate=0.012, n_estimators=2800, subsample=0.85,
                      colsample_bytree=0.8, tree_method='hist', random_state=42).fit(X, y_t, sample_weight=weights)
mo = xgb.XGBRegressor(max_depth=4, learning_rate=0.012, n_estimators=2800, subsample=0.85,
                      colsample_bytree=0.8, tree_method='hist', random_state=42).fit(X, y_o, sample_weight=weights)

prob_clf = clf.predict_proba(Xt)
lt, lo = mt.predict(Xt).clip(0.6, 3.0), mo.predict(Xt).clip(0.6, 3.0)

# ==========================================
# 4. STAGE 3 — CALIBRATED SELECTION (FIXED)
# ==========================================

def select_score_v30(lt_val, lo_val, p_win, p_draw, p_loss, elo_diff):
    best_p = -1
    best = (1, 1)


    outcome_sharpener = 1.0
    if abs(elo_diff) > 40:
        outcome_sharpener = 0.4

    for t in range(6):
        for o in range(6):
            p = (math.exp(-lt_val) * (lt_val**t) / math.factorial(t)) * \
                (math.exp(-lo_val) * (lo_val**o) / math.factorial(o))

            if t > o:   p *= (p_win ** 2)
            elif t == o: p *= (p_draw * outcome_sharpener)
            elif t < o:   p *= (p_loss ** 2)


            if abs(elo_diff) > 100 and abs(t-o) == 1:
                p *= 0.8

            if p > best_p:
                best_p, best = p, (t, o)
    return best

elo_diff_test = test['elo_diff'].values
pred = [select_score_v30(lt[i], lo[i], prob_clf[i,2], prob_clf[i,1], prob_clf[i,0], elo_diff_test[i]) for i in range(len(lt))]

print("Applying Calibrated Tier Enforcement...")
tier_diff_test = test['tier_diff'].values
pred = [select_score_v30(lt[i], lo[i], prob_clf[i,2], prob_clf[i,1], prob_clf[i,0], tier_diff_test[i]) for i in range(len(lt))]
pt, po = np.array([p[0] for p in pred]), np.array([p[1] for p in pred])

# ==========================================
# 5. FINAL CALIBRATION
# ==========================================
DRAW_RATE = (y_t == y_o).mean()
n_target = int(len(pt) * DRAW_RATE)
n_curr = (pt == po).sum()

if n_curr < n_target:
    non_idx = np.where(pt != po)[0]
    to_add = non_idx[np.argsort(np.abs(lt[non_idx]-lo[non_idx]))[:(n_target-n_curr)]]
    for i in to_add:
        avg = round((lt[i]+lo[i])/2)
        pt[i], po[i] = avg, avg
elif n_curr > n_target:
    draw_idx = np.where(pt == po)[0]
    to_rem = draw_idx[np.argsort(np.abs(lt[draw_idx]-lo[draw_idx]))[::-1][:(n_curr-n_target)]]
    for i in to_rem:
        if lt[i] >= lo[i]: pt[i] += 1
        else: po[i] += 1

test['team_goals'], test['opp_goals'] = pt.astype(int), po.astype(int)
test[['Id','team_goals','opp_goals']].to_csv("DSC26198_Data Holic_Submission.csv", index=False)

print(f"\nFINAL v29.1: W={(pt>po).mean():.3f} D={(pt==po).mean():.3f} L={(pt<po).mean():.3f}")
print(f"Gap vs Target: {(pt==po).mean() - DRAW_RATE:+.4f}")
print("DSC26198_Data Holic_Submission.csv saved")

Training Models v29.1...
Applying Calibrated Tier Enforcement...

FINAL v29.1: W=0.392 D=0.215 L=0.394
Gap vs Target: -0.0000
DSC26198_Data Holic_Submission.csv saved
